# 13 — Rangkuman Percobaan Adversarial (Paper 2)

**Untuk presentasi / PPT & untuk menyusun `paper2-adversarial.tex`.** Notebook ini
merangkai *urutan cerita* percobaan adversarial: dari fondasi Paper 1, dua sumbu
ketangguhan, empat varian model, tiga rezim serangan, hingga temuan & kesimpulan.

> **Prinsip kejujuran data:** semua angka berasal dari eksperimen nyata
> (`paper2_pipeline_meta.json` dari notebook 11, dan `paper2_eval_results.json` dari
> notebook 12). Bila berkas tersedia (lokal / diunduh dari S3), notebook memuatnya;
> bila tidak, dipakai nilai *fallback* yang identik dengan hasil tercatat sehingga
> notebook tetap jalan (mis. sebelum eksperimen dijalankan).

Jalankan sel berurutan dari atas ke bawah.

## 0. Setup & pemuatan hasil

In [ ]:
import importlib, sys, subprocess
need=[m for m in ('matplotlib','pandas','numpy') if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.grid':True,'grid.alpha':0.3})

# Cari folder hasil (lokal notebook 11/12, folder induk, atau folder out).
def find_json(name):
    cands=[name, os.path.join('paper2_eval_out',name), os.path.join('paper2_models',name),
           os.path.join('..',name), name]
    for p in cands:
        if os.path.exists(p):
            print('  loaded:', p); return json.load(open(p))
    print('  (fallback):', name); return None
print('siap.')

## 1. Latar: Dua Sumbu Ketangguhan NIDS

Paper 1 menutup **sumbu-1 (perpindahan jaringan / distribution shift)** dengan
SFM + few-shot. Paper 2 menambah **sumbu-2 (evasion adversarial)**. Pertanyaan inti:
*bisakah satu XGBoost ringan pada 9 fitur SFM tangguh di KEDUA sumbu sekaligus?*

In [ ]:
# Diagram dua sumbu ketangguhan (konsep).
fig, ax = plt.subplots(figsize=(6.4,4.6)); ax.set_aspect('equal')
ax.axhline(0,color='k',lw=0.8); ax.axvline(0,color='k',lw=0.8)
ax.set_xlim(-0.1,1.1); ax.set_ylim(-0.1,1.1)
ax.set_xlabel('Sumbu-1: generalisasi lintas-jaringan (few-shot)')
ax.set_ylabel('Sumbu-2: ketahanan evasion (adversarial)')
# Posisi NYATA (arah CIC->UNSW): sumbu-x = clean_target (generalisasi),
# sumbu-y = adaptive_functional_eps0.1 (ketahanan evasion). MCC dipetakan (v+1)/2 ke [0,1].
def nrm(v): return (v+1)/2.0
pts={'baseline':(-0.074,0.370,'#999999'),'few-shot':(0.650,0.343,'#4C72B0'),
     'adv':(-0.010,-0.440,'#DD8452'),'few-shot+adv':(0.696,0.495,'#55A868')}
for name,(gx,gy,c) in pts.items():
    x,y=nrm(gx),nrm(gy)
    ax.scatter([x],[y],s=260,color=c,edgecolor='k',zorder=3)
    ax.annotate(f'{name}\n(gen={gx:+.2f}, adv={gy:+.2f})',(x,y),textcoords='offset points',
                xytext=(0,-32),ha='center',fontsize=8)
ax.set_title('Posisi NYATA (CIC->UNSW): few-shot+adv unggul di kedua sumbu')
plt.tight_layout(); plt.show()
print('Sumbu MCC dipetakan (v+1)/2 ke [0,1]; angka asli tertera di label.')
print('few-shot+adv = kuadran kanan-atas (generalisasi 0.70 + ketahanan adaptive 0.50).')
print('=== SEL 1 (dua sumbu) SELESAI ===')

## 2. Empat Varian Model (notebook 11)

Untuk tiap arah (CIC→UNSW, UNSW→CIC), dilatih 4 varian pada 9 fitur SFM
(XGBoost biner, z-score per-dataset fit-train-only):
1. **baseline** — sumber clean.
2. **few-shot** — sumber + 1% label target.
3. **adv** — sumber + adversarial training (D_clean ∪ D_adv, rasio 20%, eps_train=0.1).
4. **few-shot+adv** — (sumber + 1% target) lalu adversarial training (usulan Paper 2).

In [ ]:
meta = find_json('paper2_pipeline_meta.json')
cfg = {'eps_train':0.1,'adv_ratio':0.20,'fewshot_frac':0.01,
       'variants':['baseline','few-shot','adv','few-shot+adv'],
       'xgboost':'max_depth=8, lr=0.1, n_estimators=200, subsample/colsample=0.8'}
if meta:
    cfg['eps_train']=meta.get('eps_train',cfg['eps_train'])
    cfg['adv_ratio']=meta.get('adv_ratio',cfg['adv_ratio'])
    cfg['fewshot_frac']=meta.get('fewshot_frac',cfg['fewshot_frac'])
print('Konfigurasi pelatihan:')
for k,v in cfg.items(): print(f'  {k}: {v}')
print('=== SEL 2 (konfigurasi varian) SELESAI ===')

## 3. Tiga Rezim Serangan (notebook 12)

Semua pada eps ∈ {0.05, 0.1, 0.2}, saliency via *central finite-difference* (h=0.01):
- **Unconstrained** — FGSM bebas di ruang z-score (batas atas daya serang; bisa flow mustahil).
- **Functional-preserving** — FGSM + proyeksi ke ruang valid protokol (non-neg; paket integer;
  bytes≥pkts; mean=bytes/pkts; monotonik add-only) — serangan yang benar-benar dapat dikirim.
- **Adaptive white-box** — saliency dari model yang diserang sendiri (skenario terburuk).

In [ ]:
regimes = pd.DataFrame([
  {'Rezim':'Unconstrained','Realistis?':'Tidak','Makna':'batas atas daya serang'},
  {'Rezim':'Functional-preserving','Realistis?':'Ya','Makna':'flow valid protokol, dapat dikirim'},
  {'Rezim':'Adaptive white-box','Realistis?':'Ya (terburuk)','Makna':'penyerang tahu pertahanan'},
])
import IPython.display as ipd; ipd.display(regimes)
print('=== SEL 3 (rezim serangan) SELESAI ===')

## 4. Hasil: Tabel MCC empat varian (clean + evasion)

Dimuat dari `paper2_eval_results.json` (notebook 12); bila berkas tak ada, memakai
*fallback* berisi **angka nyata tercatat** (hasil eksperimen), sehingga tabel tetap tampil.

In [ ]:
# Fallback = ANGKA NYATA tercatat (paper2_eval_results.json, notebook 12).
FALLBACK_ROWS=[
  {'arah':'CIC->UNSW','model':'baseline',   'clean_source':0.913,'clean_target':-0.074,'unconstrained_eps0.1':-0.128,'adaptive_functional_eps0.1':0.370},
  {'arah':'CIC->UNSW','model':'fewshot',    'clean_source':0.913,'clean_target':0.650, 'unconstrained_eps0.1':0.432, 'adaptive_functional_eps0.1':0.343},
  {'arah':'CIC->UNSW','model':'adv',        'clean_source':0.913,'clean_target':-0.010,'unconstrained_eps0.1':-0.189,'adaptive_functional_eps0.1':-0.440},
  {'arah':'CIC->UNSW','model':'fewshot_adv','clean_source':0.912,'clean_target':0.696, 'unconstrained_eps0.1':0.053, 'adaptive_functional_eps0.1':0.495},
  {'arah':'UNSW->CIC','model':'baseline',   'clean_source':0.745,'clean_target':-0.060,'unconstrained_eps0.1':0.053, 'adaptive_functional_eps0.1':-0.462},
  {'arah':'UNSW->CIC','model':'fewshot',    'clean_source':0.741,'clean_target':0.897, 'unconstrained_eps0.1':0.005, 'adaptive_functional_eps0.1':-0.121},
  {'arah':'UNSW->CIC','model':'adv',        'clean_source':0.742,'clean_target':-0.067,'unconstrained_eps0.1':-0.433,'adaptive_functional_eps0.1':0.002},
  {'arah':'UNSW->CIC','model':'fewshot_adv','clean_source':0.746,'clean_target':0.897, 'unconstrained_eps0.1':0.460, 'adaptive_functional_eps0.1':-0.025},
]
res = find_json('paper2_eval_results.json')
rows = res['rows'] if (res and res.get('rows')) else FALLBACK_ROWS
if not (res and res.get('rows')): print('(memakai FALLBACK angka nyata tertanam)')
dfe = pd.DataFrame(rows)
cols = [c for c in ['arah','model','clean_source','clean_target','unconstrained_eps0.1','adaptive_functional_eps0.1'] if c in dfe.columns]
ipd.display(dfe[cols])
print('=== SEL 4 (tabel hasil) SELESAI ===')

In [ ]:
# Grafik: generalisasi (clean_target) vs ketahanan (adaptive_functional_eps0.1) per varian.
if not dfe.empty and 'clean_target' in dfe.columns:
    for direction in dfe['arah'].unique():
        sub = dfe[dfe['arah']==direction]
        labels=sub['model'].tolist(); x=np.arange(len(labels)); w=0.38
        fig,ax=plt.subplots(figsize=(7,3.8))
        ax.bar(x-w/2, sub['clean_target'], w, label='clean (lintas-jaringan)', color='#4C72B0')
        if 'adaptive_functional_eps0.1' in sub.columns:
            ax.bar(x+w/2, sub['adaptive_functional_eps0.1'], w,
                   label='adaptive functional evasion (eps=0.1)', color='#C44E52')
        ax.axhline(0,color='k',lw=0.8); ax.set_xticks(x); ax.set_xticklabels(labels,rotation=15)
        ax.set_ylabel('MCC'); ax.set_ylim(-0.3,1.0)
        ax.set_title(f'{direction}: generalisasi vs ketahanan evasion'); ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()
else:
    print('(lewati grafik; hasil belum tersedia)')
print('=== SEL 4b (grafik hasil) SELESAI ===')

## 5. Alur Cerita (untuk narasi paper & slide)

1. **Motivasi** — NIDS rapuh di dua sumbu: perpindahan jaringan & evasion.
2. **Fondasi** — SFM + few-shot menutup sumbu-1 (Paper 1).
3. **Pertanyaan** — bisakah 1 XGBoost ringan tangguh di kedua sumbu sekaligus?
4. **Desain** — 4 varian x 2 arah; serangan finite-diff FGSM.
5. **Realisme** — bedakan unconstrained vs functional-preserving; uji adaptive white-box.
6. **Temuan (nyata)** — generalisasi HANYA dari few-shot (bukan adv); few-shot+adv tak merusak
   generalisasi; di CIC->UNSW few-shot+adv unggul DUA sumbu (gen 0.70 + adaptive 0.50), tetapi
   di UNSW->CIC ketahanan adaptive tetap runtuh (asimetri = batas adv training pd pohon).
7. **Arah lanjut** — pertahanan tahan-adaptive; integrasi adaptasi online (Paper 3).

In [ ]:
# Diagram alur cerita (7 langkah).
fig, ax = plt.subplots(figsize=(11,2.2)); ax.axis('off')
steps=['Motivasi\n2 sumbu','Fondasi\nSFM+few-shot','Pertanyaan\n1 model, 2 sumbu?',
       '4 varian\nx 2 arah','3 rezim\nserangan','Temuan\n(angka)','Arah lanjut\n(Paper 3)']
n=len(steps); x=np.linspace(0.02,0.98,n)
for i,(xi,s) in enumerate(zip(x,steps)):
    ax.add_patch(plt.Rectangle((xi-0.065,0.35),0.13,0.3,fc='#EAF0F7',ec='#4C72B0',lw=1.5))
    ax.text(xi,0.5,s,ha='center',va='center',fontsize=8)
    if i<n-1:
        ax.annotate('',xy=(x[i+1]-0.07,0.5),xytext=(xi+0.07,0.5),
                    arrowprops=dict(arrowstyle='->',color='#333',lw=1.3))
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title('Alur cerita percobaan adversarial (Paper 2)',fontsize=11)
plt.tight_layout(); plt.show()
print('=== SEL 5 (alur cerita) SELESAI ===')

## 6. Rangkuman Temuan Utama (slide penutup) — ANGKA NYATA

- **Generalisasi hanya dari few-shot, bukan adv.** Varian *adv* gagal lintas-jaringan
  (MCC target $-0{,}01$ / $-0{,}07$); *few-shot* & *few-shot+adv* pulih ke $0{,}65$–$0{,}90$.
- **Adv training tidak merusak generalisasi.** *few-shot+adv* setara *few-shot* di clean-target
  ($0{,}696$ vs $0{,}650$ CIC$\to$UNSW; $0{,}897$ vs $0{,}897$ UNSW$\to$CIC).
- **CIC$\to$UNSW: few-shot+adv unggul DUA sumbu** — generalisasi $0{,}696$ + ketahanan
  *adaptive functional evasion* $0{,}495$ (jauh di atas *adv* yang runtuh $-0{,}44$).
- **UNSW$\to$CIC: ketahanan adaptive tetap runtuh** untuk semua varian ($\le 0{,}00$) —
  temuan asimetri = batas *adversarial training* pada model pohon di bawah *adaptive white-box*.
- Serangan *unconstrained* melebih-lebihkan ancaman dibanding *functional-preserving* (lebih realistis).
- Metrik utama **MCC** (tahan class imbalance), konsisten dengan Paper 1.

*Satu keberhasilan (CIC$\to$UNSW) dan satu batas (UNSW$\to$CIC) — keduanya kontribusi jujur.*